In [1]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import time

In [16]:
arr = np.load('resultados\gamma_gamma-1_MS-3.0.npy')

data = np.column_stack((arr[:,0], arr[:,2]))
data = data[~np.isnan(data).any(axis=1)]

# Crear DataFrame
df = pd.DataFrame(data, columns=['label', 'SBP'])

# Guardar CSV sin notación científica
df.to_csv('datos_limpios.csv', index=False, float_format='%.10f')

In [1]:
# GraficaRRySBP.py
# Traducción de código MATLAB
# Por: Claudia Lerma
# Última actualización: Marzo 25, 2025 (versión traducida a Python)

import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import time

start_time = time.time()

# Rutas
RutaDatos = r''
RutaFiguras = r'Figuras\\'

# Leer nombres de archivos y códigos
archivos = np.loadtxt(os.path.join(RutaDatos, 'RecordingNames.txt'), dtype=str)
claves = np.loadtxt(os.path.join(RutaDatos, 'Codes.txt'))

Nrec = len(claves)
Npuntos = 700
Ndatos = np.zeros((Nrec, 2), dtype=int)

for registro in range(Nrec):
    filtro = claves[:, 3]
    if filtro[registro] == 0:
        # Leer RR y SBP
        arch = os.path.join(RutaDatos, 'RRnSBP15min', archivos[registro] + 'RRnSBP15min.txt')
        Series = np.loadtxt(arch)

        tRR, RRcomplete = Series[:, 0], Series[:, 1]
        tSBP, SBP = Series[:, 2], Series[:, 3]

        Ndatos[registro, 0] = len(tRR)

        if len(tRR) >= Npuntos:
            tRRc, RRc = tRR[:Npuntos], RRcomplete[:Npuntos]
            tSBPc, SBPc = tSBP[:Npuntos], SBP[:Npuntos]
        else:
            tRRc, RRc = tRR, RRcomplete
            tSBPc, SBPc = tSBP, SBP

        Ndatos[registro, 1] = len(RRc)

        arch2 = os.path.join(RutaDatos,'DatosRRnSBP700' ,archivos[registro] + 'RRnSBP700.txt')
        Series_cortada = np.column_stack((tRRc, RRc, tSBPc, SBPc))
        np.savetxt(arch2, Series_cortada, fmt='%.6f')

        # Graficar
        fig, ax = plt.subplots(2, 1, figsize=(10, 6), sharex=True)

        ax[0].plot(tRR, RRcomplete, '.-k')
        ax[0].plot(tRRc, RRc, '-r')
        ax[0].set_title(archivos[registro])
        ax[0].set_ylabel('RR interval (s)')
        ax[0].grid(True)

        ax[1].plot(tSBP, SBP, '.-k')
        ax[1].plot(tSBPc, SBPc, '-r')
        ax[1].set_ylabel('SBP (mmHg)')
        ax[1].set_xlabel('Time (s)')
        ax[1].grid(True)

        fig.tight_layout()
        fig.savefig(os.path.join(RutaFiguras, archivos[registro] + 'RRnSBP.png'), dpi=300)
        plt.close(fig)

    else:
        print(f'CHECK: {archivos[registro]} descartado por ruido, arritmia, etc.')

# Guardar resultados
archResultados = os.path.join(RutaDatos, 'NlatidosBetas.txt')
np.savetxt(archResultados, Ndatos, fmt='%d')

print(f"Tiempo de ejecución: {time.time() - start_time:.2f} segundos")


CHECK: 0057 descartado por ruido, arritmia, etc.
CHECK: 0091 descartado por ruido, arritmia, etc.
CHECK: 0092 descartado por ruido, arritmia, etc.
CHECK: 0130 descartado por ruido, arritmia, etc.
CHECK: 0167 descartado por ruido, arritmia, etc.
CHECK: 0172 descartado por ruido, arritmia, etc.
CHECK: 0179 descartado por ruido, arritmia, etc.
CHECK: 0185 descartado por ruido, arritmia, etc.
CHECK: 0186 descartado por ruido, arritmia, etc.
CHECK: 0218 descartado por ruido, arritmia, etc.
CHECK: 0219 descartado por ruido, arritmia, etc.
CHECK: 0238 descartado por ruido, arritmia, etc.
CHECK: 0239 descartado por ruido, arritmia, etc.
CHECK: 0244 descartado por ruido, arritmia, etc.
CHECK: 0249 descartado por ruido, arritmia, etc.
CHECK: 0252 descartado por ruido, arritmia, etc.
CHECK: 0272 descartado por ruido, arritmia, etc.
CHECK: 0273 descartado por ruido, arritmia, etc.
CHECK: 0276 descartado por ruido, arritmia, etc.
CHECK: 0278 descartado por ruido, arritmia, etc.
CHECK: 0299 descarta

In [37]:
data = np.load('resultados\gamma_0003_gamma-1_MS-3.0.npy', allow_pickle=True).item()

print(list(data.values())[0])

0.9000039168272792


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import pearsonr
import os

MS = 3.0
method = 'lineal'
size = 0
indice_gamma = 1

# ----------------------------------------------------------------------
# Ajusta estos nombres si cambiaste los parámetros de salida
# ----------------------------------------------------------------------
FILE_J     = f'resultados/J_method-{method}_size-{size}.npy'
FILE_GAMMA = f'resultados/gamma_gamma-{indice_gamma}_MS-{MS}.npy'

# ----------------------------------------------------------------------
# Cargar matrices:  (filas = pacientes 0001-1121)
#   col 0 → ID (int)
#   col 1 → RR
#   col 2 → SBP
# ----------------------------------------------------------------------
J_mat     = np.load(FILE_J)        # shape (1121, 3)
gamma_mat = np.load(FILE_GAMMA)    # shape (1121, 3)

# Índice rápido: ID (string '0001') → fila
id_to_idx = {f'{int(row[0]):04d}': i for i, row in enumerate(J_mat)}

# ----------------------------------------------------------------------
# Umbral J mínimo continuo (N = 700  ⇒ N/2 = 350)
# ----------------------------------------------------------------------
J_min = np.load('J_minus_continuo.npy')
idx_thr = np.where(J_min[0] == 350)[0]
J_crit  = J_min[1, idx_thr[0]] if idx_thr.size else np.nan

# ----------------------------------------------------------------------
# Cargar sexos
# ----------------------------------------------------------------------
ids_df  = pd.read_csv('ids.csv', dtype=str).fillna('')
mujeres = [pid.zfill(4) for pid in ids_df['M'] if pid]
hombres = [pid.zfill(4) for pid in ids_df['H'] if pid]
todos   = sorted(set(mujeres + hombres))

grupos  = {'M': mujeres, 'H': hombres, 'Todos': todos}
series  = {'RR': 1, 'SBP': 2}      # columna de la matriz

# ----------------------------------------------------------------------
# Análisis
# ----------------------------------------------------------------------
for serie, col in series.items():
    datos_J, datos_G   = {g: [] for g in grupos}, {g: [] for g in grupos}
    bajo_umbral, n_obs = {g: 0  for g in grupos}, {g: 0  for g in grupos}
    r_pearson          = {}

    # --- recopilar valores ----------------------------------------------------
    for g, ids in grupos.items():
        for pid in ids:
            idx = id_to_idx.get(pid)
            if idx is None:
                continue           # paciente no procesado
            J_val = J_mat[idx, col]
            G_val = gamma_mat[idx, col]
            if not (np.isnan(J_val) or np.isnan(G_val)):
                datos_J[g].append(J_val)
                datos_G[g].append(G_val)
                n_obs[g] += 1
                if J_val < J_crit:
                    bajo_umbral[g] += 1

        # correlación por grupo
        if len(datos_J[g]) >= 2:
            r_pearson[g] = pearsonr(datos_J[g], datos_G[g])[0]
        else:
            r_pearson[g] = np.nan

    # --- boxplot --------------------------------------------------------------
    fig, ax = plt.subplots(figsize=(10, 6))
    pos = np.arange(len(grupos))
    ancho = 0.35

    # boxplots J
    bp_J = ax.boxplot([datos_J[g] for g in grupos],
                      positions=pos - ancho/2, widths=0.25,
                      patch_artist=True, boxprops=dict(facecolor='skyblue'))
    # boxplots gamma
    bp_G = ax.boxplot([datos_G[g] for g in grupos],
                      positions=pos + ancho/2, widths=0.25,
                      patch_artist=True, boxprops=dict(facecolor='lightcoral'))

    # etiquetas con % < Jcrit
    etiquetas = []
    for g in grupos:
        pct = 100*bajo_umbral[g]/n_obs[g] if n_obs[g] else 0
        etiquetas.append(f"{g} ({pct:.1f}% < Jcrit)")

    ax.set_xticks(pos)
    ax.set_xticklabels(etiquetas)
    ax.set_title(f"Boxplots de J (method-{method}_size-{size}) y gamma (gamma-{indice_gamma}_MS-{MS}) para {serie}")
    ax.legend([bp_J['boxes'][0], bp_G['boxes'][0]], ['J', 'gamma'], loc='upper right')
    ax.set_ylabel("Valor")
    plt.tight_layout()
    plt.savefig(f"boxplot_{serie}.png", dpi=300)
    plt.close()

    # --- imprimir correlaciones ----------------------------------------------
    print(f"\nCorrelaciones de Pearson J vs gamma – {serie}")
    for g in grupos:
        print(f"  {g}: {r_pearson[g]:.3f}")



Correlaciones de Pearson J vs gamma – RR
  M: 0.064
  H: 0.203
  Todos: 0.151

Correlaciones de Pearson J vs gamma – SBP
  M: 0.470
  H: 0.291
  Todos: 0.371


In [5]:
def calcular_gamma_opt(data, gamma_index, MS):
    N = len(data)
    sd = np.std(data, ddof=1)
    eps = sd / MS
    maxdat = np.max(data)
    data = np.concatenate(([0.0], data, [maxdat + 100 * eps]))  # data[0], data[N+1]

    Ci = [0] * (gamma_index + 2)  # Necesitamos Ci[i-1], Ci[i], Ci[i+1]

    for j in range(1, N + 1):
        for i in range(1, j):
            k = 0
            while k <= gamma_index + 1 and abs(data[i + k] - data[j + k]) <= eps:
                if k in (gamma_index - 1, gamma_index, gamma_index + 1):
                    Ci[k] += 1
                k += 1

    norm = 2.0 / (N * (N - 1))
    C = [1.0] + [0.0] * (gamma_index + 1)
    for k in (gamma_index - 1, gamma_index, gamma_index + 1):
        if k >= 1:
            C[k] = Ci[k] * norm

    denominator = C[gamma_index - 1] * C[gamma_index + 1]
    gamma = 0.0
    if denominator != 0:
        gamma = 1.0 - (C[gamma_index] ** 2) / denominator

    return gamma

In [108]:
J_univariante(np.loadtxt('DatosRRnSBP700/0268RRnSBP700.txt')[:,3])

0.9228739864144342

In [9]:
calcular_gamma_opt(np.loadtxt('DatosRRnSBP700/0004RRnSBP700.txt')[:,3],1,3.0)

0.7697114467064519

In [ ]:
import pandas as pd

indx_mujer = pd.read_excel('CRPagingBetas.xlsx')['M'].dropna().astype(int).to_numpy() - 1

indx_mujer
# J_mat[9,1]

array([   3,    9,   12,   13,   15,   17,   20,   21,   22,   23,   25,
         27,   31,   34,   36,   38,   40,   45,   46,   48,   53,   54,
         61,   65,   70,   71,   72,   74,   75,   78,   82,   83,   85,
         86,   87,   88,   96,   99,  100,  101,  102,  103,  108,  110,
        111,  112,  114,  115,  121,  122,  123,  128,  132,  136,  139,
        142,  143,  145,  146,  150,  153,  156,  157,  158,  160,  162,
        167,  169,  173,  174,  175,  182,  186,  190,  192,  193,  194,
        195,  198,  207,  208,  209,  210,  211,  216,  219,  220,  223,
        224,  225,  226,  227,  231,  233,  244,  249,  254,  255,  256,
        260,  262,  265,  267,  273,  278,  279,  280,  282,  284,  292,
        296,  300,  302,  307,  315,  316,  318,  326,  335,  338,  343,
        345,  347,  348,  349,  353,  354,  355,  358,  359,  366,  367,
        370,  374,  379,  380,  381,  388,  389,  390,  392,  397,  403,
        404,  405,  406,  410,  417,  418,  420,  4

In [ ]:
data = pd.read_excel('CRPagingBetas.xlsx')[['Age_group','Filename']]
data[data['Age_group'] == 2.0]['Filename'].dropna().astype(int).to_numpy() - 1

array([   0,    1,    4,    6,    9,   10,   17,   18,   19,   24,   25,
         27,   30,   32,   39,   40,   41,   42,   45,   54,   55,   57,
         58,   62,   63,   64,   65,   66,   70,   83,   87,   88,   89,
         93,   94,   96,  104,  108,  109,  114,  115,  117,  118,  120,
        124,  126,  127,  134,  137,  141,  144,  148,  159,  166,  168,
        176,  178,  182,  183,  189,  192,  196,  200,  202,  204,  205,
        208,  212,  215,  221,  222,  224,  225,  234,  235,  240,  242,
        245,  252,  256,  258,  259,  265,  268,  269,  270,  271,  276,
        281,  282,  283,  285,  287,  288,  290,  294,  299,  304,  307,
        309,  310,  312,  317,  318,  319,  321,  324,  325,  328,  329,
        330,  333,  338,  339,  340,  342,  343,  351,  358,  360,  361,
        362,  366,  370,  374,  376,  377,  378,  382,  384,  387,  388,
        393,  396,  397,  399,  401,  402,  406,  407,  410,  415,  416,
        418,  421,  423,  424,  425,  431,  432,  4

In [36]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import pearsonr
import os

# ════════════════════════════════════════════════════════════════════════
# Parámetros que ya usabas
# ════════════════════════════════════════════════════════════════════════
MS = 3.0
method = 'lineal'
size = 0
indice_gamma = 1

FILE_J     = f'resultados/J_method-{method}_size-{size}.npy'
FILE_GAMMA = f'resultados/gamma_gamma-{indice_gamma}_MS-{MS}.npy'

# ════════════════════════════════════════════════════════════════════════
# Cargar matrices J y γ  (filas = pacientes 0001-1121, columnas = ID, RR, SBP)
# ════════════════════════════════════════════════════════════════════════
J_mat     = np.load(FILE_J)        # shape (1121, 3)
gamma_mat = np.load(FILE_GAMMA)    # shape (1121, 3)

# Mapeo rápido: '0001' → fila correspondiente
id_to_idx = {f'{int(row[0]):04d}': i for i, row in enumerate(J_mat)}

# ════════════════════════════════════════════════════════════════════════
# Umbral J crítico (misma lógica que antes)
# ════════════════════════════════════════════════════════════════════════
J_min    = np.load('J_minus_continuo.npy')         # [2 × …]
idx_thr  = np.where(J_min[0] == 350)[0]
J_crit   = J_min[1, idx_thr[0]] if idx_thr.size else np.nan

# ════════════════════════════════════════════════════════════════════════
# CONSTRUIR LOS GRUPOS POR EDAD -----------------------------------------
#   • Se parte del archivo CRPagingBetas.xlsx
#   • Age_group suele codificarse 1.0, 2.0, 3.0, …
#   • Filename trae el número de paciente (0001–1121)
# ════════════════════════════════════════════════════════════════════════
age_df = (
    pd.read_excel('CRPagingBetas.xlsx', usecols=['Age_group', 'Filename'])
      .dropna(subset=['Filename'])                         # quitamos filas sin ID
)

# Aseguramos que Age_group sea entero y los IDs estén en formato '0001'

age_df = (
    pd.read_excel('CRPagingBetas.xlsx', usecols=['Age_group', 'Filename'])
    .dropna(subset=['Filename', 'Age_group'])  # eliminamos también los que no tienen grupo
)

age_df['Age_group'] = age_df['Age_group'].astype(int)
age_df['ID'] = age_df['Filename'].astype(int).astype(str).str.zfill(4)

age_df['Age_group'] = age_df['Age_group'].astype(int)
age_df['ID']        = age_df['Filename'].astype(int).astype(str).str.zfill(4)

# Diccionario { 'grupo1': [id1, id2, …], 'grupo2': […], …, 'Todos': […] }
grupos = {f'grupo{g}': age_df.loc[age_df['Age_group'] == g, 'ID'].tolist()
          for g in sorted(age_df['Age_group'].unique())}

grupos['Todos'] = sorted(set(age_df['ID']))

# ════════════════════════════════════════════════════════════════════════
# Serie  ↔  columna dentro de las matrices cargadas
# ════════════════════════════════════════════════════════════════════════
series = {'RR': 1, 'SBP': 2}

# ════════════════════════════════════════════════════════════════════════
# Análisis (idéntico al original, solo cambian los grupos)
# ════════════════════════════════════════════════════════════════════════
for serie, col in series.items():
    datos_J, datos_G   = {g: [] for g in grupos}, {g: [] for g in grupos}
    bajo_umbral, n_obs = {g: 0  for g in grupos}, {g: 0  for g in grupos}
    r_pearson          = {}

    # ─── recopilar valores ───────────────────────────────────────────────
    for g, ids in grupos.items():
        for pid in ids:
            idx = id_to_idx.get(pid)
            if idx is None:               # paciente no procesado
                continue
            J_val = J_mat[idx, col]
            G_val = gamma_mat[idx, col]
            if not (np.isnan(J_val) or np.isnan(G_val)):
                datos_J[g].append(J_val)
                datos_G[g].append(G_val)
                n_obs[g] += 1
                if J_val < J_crit:
                    bajo_umbral[g] += 1

        # Correlación J vs γ para ese grupo
        if len(datos_J[g]) >= 2:
            r_pearson[g] = pearsonr(datos_J[g], datos_G[g])[0]
        else:
            r_pearson[g] = np.nan

    # ─── boxplot ─────────────────────────────────────────────────────────
    fig, ax = plt.subplots(figsize=(10, 6))
    pos = np.arange(len(grupos))
    ancho = 0.35

    # J
    bp_J = ax.boxplot([datos_J[g] for g in grupos],
                      positions=pos - ancho/2, widths=0.25,
                      patch_artist=True,
                      boxprops=dict(facecolor='skyblue'))
    # γ
    bp_G = ax.boxplot([datos_G[g] for g in grupos],
                      positions=pos + ancho/2, widths=0.25,
                      patch_artist=True,
                      boxprops=dict(facecolor='lightcoral'))

    # Etiquetas: «grupoX (yy % < Jcrit)»
    etiquetas = []
    for g in grupos:
        pct = 100 * bajo_umbral[g] / n_obs[g] if n_obs[g] else 0
        etiquetas.append(f"{g} ({pct:.1f}% < Jcrit)")

    ax.set_xticks(pos)
    ax.set_xticklabels(etiquetas, rotation=15)
    ax.set_title(f"Boxplots J (method-{method}_size-{size}) y Gamma (Gamma-{indice_gamma}_MS-{MS}) – {serie}")
    ax.legend([bp_J['boxes'][0], bp_G['boxes'][0]], ['J', 'Gamma'], loc='upper right')
    ax.set_ylabel("Valor")
    plt.tight_layout()
    plt.savefig(f"boxplot_{serie}_edad.png", dpi=300)
    plt.close()

    # ─── imprimir correlaciones ─────────────────────────────────────────
    print(f"\nCorrelaciones de Pearson J vs γ – {serie}")
    for g in grupos:
        print(f"  {g}: {r_pearson[g]:.3f}")



Correlaciones de Pearson J vs γ – RR
  grupo1: -0.039
  grupo2: 0.154
  grupo3: 0.095
  grupo4: -0.031
  grupo5: -0.196
  grupo6: 0.090
  grupo7: -0.083
  grupo8: 0.492
  grupo9: 0.098
  grupo10: 0.209
  grupo11: 0.202
  grupo12: -0.179
  grupo13: 0.699
  grupo14: -0.522
  grupo15: 0.550
  Todos: 0.141

Correlaciones de Pearson J vs γ – SBP
  grupo1: 0.489
  grupo2: 0.492
  grupo3: 0.345
  grupo4: 0.289
  grupo5: -0.070
  grupo6: 0.262
  grupo7: 0.159
  grupo8: -0.031
  grupo9: -0.408
  grupo10: -0.015
  grupo11: 0.397
  grupo12: -0.098
  grupo13: 0.242
  grupo14: 0.528
  grupo15: -0.008
  Todos: 0.370


In [18]:

def J_univariante(X):
    def distancia(p1, p2):
        return np.linalg.norm(np.array(p2) - np.array(p1))
    X = np.array(X)
    x1 = X[1:]
    y1 = X[:-1]
    ff1 = np.angle(np.fft.rfft(x1))
    ff2 = np.angle(np.fft.rfft(y1))

    vectores = []
    for i in range(len(ff1) - 1):
        p1 = [ff1[i], ff2[i]]
        p2 = [ff1[i + 1], ff2[i + 1]]
        cuadrante = [
            [p2[0] - p1[0], p2[1] - p1[1]],
            [p2[0] - p1[0], p2[1] + 2 * np.pi - p1[1]],
            [p2[0] + 2 * np.pi - p1[0], p2[1] + 2 * np.pi - p1[1]],
            [p2[0] + 2 * np.pi - p1[0], p2[1] - p1[1]],
            [p2[0] + 2 * np.pi - p1[0], p2[1] - 2 * np.pi - p1[1]],
            [p2[0] - p1[0], p2[1] - 2 * np.pi - p1[1]],
            [p2[0] - 2 * np.pi - p1[0], p2[1] - 2 * np.pi - p1[1]],
            [p2[0] - 2 * np.pi - p1[0], p2[1] - p1[1]],
            [p2[0] - 2 * np.pi - p1[0], p2[1] + 2 * np.pi - p1[1]],
        ]
        distancias = np.array([distancia(p1, c) for c in cuadrante])
        p2 = cuadrante[np.argmin(distancias)]
        vectores.append([p2[0] - p1[0], p2[1] - p1[1]])

    vectores = np.array(vectores)
    norms = np.linalg.norm(vectores, axis=1, keepdims=True)
    v_norm = np.where(norms == 0, vectores, vectores / norms)
    
    angulos = np.arccos(np.clip(np.einsum('ij,ij->i', v_norm[:-1], v_norm[1:]), -1.0, 1.0))
    cruces = np.cross(v_norm[:-1], v_norm[1:])
    angulos = np.where(cruces > 0, np.pi - angulos, angulos)
    angulos = np.where((cruces == 0) & (angulos < 0), np.pi, angulos)
    angulos = np.where(cruces < 0, angulos + np.pi, angulos)

    e = np.exp(angulos * 1j)
    e1 = np.sum(e) / len(angulos)
    J = 1.0 - np.abs(e1.real)
    
    return J

def J_bivariante(X, Y, corte):
    def distancia(p1, p2):
        return np.linalg.norm(np.array(p2) - np.array(p1))
    X = np.array(X)
    x1 = X[:]
    y1 = Y[:]
    ff1 = np.angle(np.fft.rfft(x1))
    ff2 = np.angle(np.fft.rfft(y1))

    vectores = []
    for i in range(len(ff1) - 1):
        p1 = [ff1[i], ff2[i]]
        p2 = [ff1[i + 1], ff2[i + 1]]
        cuadrante = [
            [p2[0] - p1[0], p2[1] - p1[1]],
            [p2[0] - p1[0], p2[1] + 2 * np.pi - p1[1]],
            [p2[0] + 2 * np.pi - p1[0], p2[1] + 2 * np.pi - p1[1]],
            [p2[0] + 2 * np.pi - p1[0], p2[1] - p1[1]],
            [p2[0] + 2 * np.pi - p1[0], p2[1] - 2 * np.pi - p1[1]],
            [p2[0] - p1[0], p2[1] - 2 * np.pi - p1[1]],
            [p2[0] - 2 * np.pi - p1[0], p2[1] - 2 * np.pi - p1[1]],
            [p2[0] - 2 * np.pi - p1[0], p2[1] - p1[1]],
            [p2[0] - 2 * np.pi - p1[0], p2[1] + 2 * np.pi - p1[1]],
        ]
        distancias = np.array([distancia(p1, c) for c in cuadrante])
        p2 = cuadrante[np.argmin(distancias)]
        vectores.append([p2[0] - p1[0], p2[1] - p1[1]])

    vectores = np.array(vectores)
    norms = np.linalg.norm(vectores, axis=1, keepdims=True)
    v_norm = np.where(norms == 0, vectores, vectores / norms)
    
    angulos = np.arccos(np.clip(np.einsum('ij,ij->i', v_norm[:-1], v_norm[1:]), -1.0, 1.0))
    cruces = np.cross(v_norm[:-1], v_norm[1:])
    angulos = np.where(cruces > 0, np.pi - angulos, angulos)
    angulos = np.where((cruces == 0) & (angulos < 0), np.pi, angulos)
    angulos = np.where(cruces < 0, angulos + np.pi, angulos)

    e = np.exp(angulos * 1j)
    e1 = np.sum(e) / len(angulos)
    J = 1.0 - np.abs(e1.real)
    
    return J